In [0]:
# %pip install groq

In [0]:
# import ast
# import re

# from groq import Groq
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col
# from typing import Dict, Any, List, Optional, Tuple

# class PolicyExtractor:

#     def __init__(self):
#         self.client = Groq(api_key="")
#         self.EMAIL_REGEX = r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b"
#         self.PHONE_REGEX = r"\b(?:\+?\d{1,3}[\s\-]?)?(?:\(?\d{2,4}\)?[\s\-]?)?\d{3,4}[\s\-]?\d{4}\b"
#         self.ID_REGEX = r"\b\d{13}\b"
#         self.POLICY_REGEX = r"\b[A-Z0-9\-]{6,25}\b"

#     def extract_identifiers_llm(self, text: str) -> Dict[str, List[str]]:

#         completion = self.client.chat.completions.create(
#             model="llama-3.3-70b-versatile",
#             messages=[
#                 {
#                     "role": "user",
#                     "content": f"""# Instructions
#             Given the below conversation history, identify all possible:

#             - email addresses
#             - cellphone numbers
#             - South African ID numbers
#             - policy numbers

#             **NOTE** If the user / customer provides the correct policy number after a disambiguation messgae, only respond with the policy number.

#             # Response format

#             Respond with **only** valid JSON in the following format:

#             {{
#                 "emails": [],
#                 "phones": [],
#                 "ids": [],
#                 "policy_numbers": []
#             }}

#             # Conversation history

#             {text}
#             """
#                 }
#             ],
#             temperature=0,
#             max_completion_tokens=2048,
#             top_p=1,
#             stream=False,
#             response_format={"type": "json_object"},
#             stop=None
#         )

#         return ast.literal_eval(completion.choices[0].message.content)

    
#     def extract_identifiers(self, text: str) -> Dict[str, List[str]]:

#         emails = list(set(re.findall(self.EMAIL_REGEX, text, flags=re.IGNORECASE)))

#         phones = list(set(
#             re.sub(r"[^\d+]", "", p)
#             for p in re.findall(self.PHONE_REGEX, text)
#         ))

#         ids = list(set(re.findall(self.ID_REGEX, text)))

#         possible_policies = list(set(
#             p.upper()
#             for p in re.findall(self.POLICY_REGEX, text)
#         ))

#         return {
#             "emails": emails,
#             "phones": phones,
#             "ids": ids,
#             "policy_numbers": possible_policies
#         }

#     def extract_policies(self, text: str) -> List[str]:

#         # identifiers = self.extract_identifiers(text)
#         identifiers = self.extract_identifiers_llm(text)
#         found = []

#         spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

#         df = spark.table(
#             "classic_demo.insurance_customer_service.customer_identifiers"
#         )

#         identifiers_to_find = (
#             identifiers["emails"] +
#             identifiers["phones"] +
#             identifiers["ids"] +
#             identifiers["policy_numbers"]
#         )

#         result = (
#             df.filter(col("identifier").isin(identifiers_to_find) | col("policy_number").isin(identifiers_to_find))
#             .select("policy_number")
#             .distinct()
#         )

#         found = [r.policy_number for r in result.collect()]

#         return found

#     def resolve_policy_context(self, text: str, customer_identifier: Optional[str] = None) -> Dict[str, Any]:
#         """
#         Disambiguate policy context from customer correspondence.
        
#         Returns:
#             Dict containing status:
#             - 'SINGLE_POLICY_FOUND': Exactly 1 policy identified.
#             - 'MULTIPLE_POLICIES_AMBIGUOUS': Multiple policies found; requires user clarification.
#             - 'NO_POLICY_FOUND': No policy identified; prompts user to provide policy number.
#         """

#         extracted = self.extract_policies(text)
        
#         if len(extracted) == 1:
#             pol_num = extracted[0]
#             # pol = db_store.get_policy(pol_num)
#             pol = pol_num
#             return {
#                 "status": "SINGLE_POLICY_FOUND",
#                 "policy_number": pol_num,
#                 "policy_details": pol,
#                 "needs_disambiguation": False,
#                 "message": f"Identified policy {pol_num}."
#             }
            
#         elif len(extracted) > 1:
#             policies_info = []
#             for pol_num in extracted:
#                 # pol = db_store.get_policy(pol_num)
#                 pol = pol_num
#                 if pol:
#                     policies_info.append({
#                         "policy_number": pol_num,
#                         # "product_type": pol["product_type"],
#                         # "status": pol["status"]
#                         "product_type": "Unknown Product", 
#                         "status": "Unknown"
#                     })
#                 else:
#                     policies_info.append(
#                         {
#                             "policy_number": pol_num, 
#                             "product_type": "Unknown Product", 
#                             "status": "Unknown"
#                          }
#                     )
                    
#             options_text = "\n".join([f"  • {p['policy_number']} - {p['product_type']} ({p['status']})" for p in policies_info])
            
#             return {
#                 "status": "MULTIPLE_POLICIES_AMBIGUOUS",
#                 "extracted_policies": extracted,
#                 "policies_info": policies_info,
#                 "needs_disambiguation": True,
#                 "disambiguation_prompt": (
#                     f"We found multiple active policies associated with your request:\n{options_text}\n\n"
#                     f"Please specify which policy number you would like us to apply this request to (e.g. '{extracted[0]}')."
#                 ),
#                 "message": f"Multiple policies detected ({', '.join(extracted)}). Disambiguation required."
#             }
            
#         else:
#             return {
#                 "status": "NO_POLICY_FOUND",
#                 "needs_disambiguation": True,
#                 "disambiguation_prompt": (
#                     "Could you please provide your policy number (e.g., POL-1001) so I can access your details and assist you?"
#                 ),
#                 "message": "No policy number identified in message."
#             }

#     def get_debit_order_info(self, policy_number: str) -> Dict[str, Any]:

#         spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
#         df = spark.table("classic_demo.insurance_customer_service.policies.")
#         policy = df.filter(col("policy_number") == policy_number).collect()

#         df = spark.table("classic_demo.insurance_customer_service.debit_orders")
#         debit_info = df.filter(col("policy_number") == policy_number).collect()

#         if not debit_info:
#             return {"success": False, "message": f"No debit order instruction found for policy '{policy_number}'."}

#         return {
#             "success": True,
#             "policy_number": policy_number,
#             "account_holder": debit_info["account_holder"],
#             "bank_name": debit_info["bank_name"],
#             "account_number_masked": debit_info["account_number_masked"],
#             "branch_code": debit_info["branch_code"],
#             "debit_day": f"Day {debit_info['debit_day']} of each month",
#             "debit_amount": f"{policy['currency']} {debit_info['debit_amount']:.2f}",
#             "last_successful_debit_date": debit_info["last_successful_debit_date"],
#             "last_debit_status": debit_info["last_debit_status"],
#             "next_debit_date": debit_info["next_debit_date"],
#             "payment_frequency": debit_info["payment_frequency"],
#             "message": f"Debit order for policy {policy_number} is scheduled for day {debit_info['debit_day']} of every month at R {debit_info['debit_amount']:.2f}."
#         }

#     def get_policy_summary(self, policy_number: str) -> Dict[str, Any]:

#         spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
#         df = spark.table("classic_demo.insurance_customer_service.policies")
#         policy = df.filter(col("policy_number") == policy_number).collect()
#         if not policy:
#             return {"success": False, "message": f"Policy '{policy_number}' not found."}
            
#         df = spark.table("classic_demo.insurance_customer_service.policies")
#         cust = df.filter(col("customer_id") == policy["customer_id"]).collect()

#         df = spark.table("classic_demo.insurance_customer_service.policies")
#         vehicles = df.filter(col("policy_number") == policy_number).collect()
        
#         return {
#             "success": True,
#             "policy_number": policy_number,
#             # "policyholder_name": f"{cust.get('first_name', '')} {cust.get('last_name', '')}",
#             "product_type": policy["product_type"],
#             "status": policy["status"],
#             "start_date": policy["start_date"],
#             "premium_amount": f"{policy['currency']} {policy['premium_amount']:.2f}",
#             "basic_excess": f"{policy['currency']} {policy['excess_amount']:.2f}",
#             "cover_details": policy["cover_details"],
#             "noted_vehicles": [f"{v['year']} {v['make']} {v['model']} ({v['registration_number']})" for v in vehicles] if vehicles else ["None"]
#         }

#     def get_summary(self, possible_policy_numbers: List[str]) -> str:

#         if possible_policy_numbers:
#             summary = ""
#             for policy in possible_policy_numbers:
#                 summary += f"\n#### Policy: {policy}"
#                 summary += f"\n{self.get_policy_summary(policy)}"
#                 summary += f"\n\n{self.get_debit_order_info(policy)}"
#                 summary += "\n\n================================================="
        
#         else:
#             summary = 'No policies could be identified at the moment - waiting for the system to respond'

#         return summary

# policy_extractor = PolicyExtractor()


In [0]:
# message = 'Hello, POL-100001'
# print(policy_extractor.extract_identifiers_llm(message))
# print("="*100)
# print(policy_extractor.extract_policies(message))
# print("="*100)
# print(policy_extractor.resolve_policy_context(message))



In [0]:
import ast
import re

from groq import Groq
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from typing import Dict, Any, List, Optional, Tuple

class PolicyExtractor:

    def __init__(self):
        self.client = Groq(api_key="")
        self.EMAIL_REGEX = r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b"
        self.PHONE_REGEX = r"\b(?:\+?\d{1,3}[\s\-]?)?(?:\(?\d{2,4}\)?[\s\-]?)?\d{3,4}[\s\-]?\d{4}\b"
        self.ID_REGEX = r"\b\d{13}\b"
        self.POLICY_REGEX = r"\b[A-Z0-9\-]{6,25}\b"

    def extract_identifiers_llm(self, text: str) -> Dict[str, List[str]]:

        completion = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": f"""# Instructions
            Given the below conversation history, identify all possible:

            - email addresses
            - cellphone numbers
            - South African ID numbers
            - policy numbers

            **NOTE** If the user / customer provides the correct policy number after a disambiguation messgae, only respond with the policy number.

            # Response format

            Respond with **only** valid JSON in the following format:

            {{
                "emails": [],
                "phones": [],
                "ids": [],
                "policy_numbers": []
            }}

            # Conversation history

            {text}
            """
                }
            ],
            temperature=0,
            max_completion_tokens=2048,
            top_p=1,
            stream=False,
            response_format={"type": "json_object"},
            stop=None
        )

        return ast.literal_eval(completion.choices[0].message.content)

    def extract_policies(self, text: str) -> List[str]:

        # identifiers = self.extract_identifiers(text)
        identifiers = self.extract_identifiers_llm(text)
        found = []

        spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

        df = spark.table(
            "classic_demo.insurance_customer_service.customer_identifiers"
        )

        identifiers_to_find = (
            identifiers["emails"] +
            identifiers["phones"] +
            identifiers["ids"] +
            identifiers["policy_numbers"]
        )

        result = (
            df.filter(col("identifier").isin(identifiers_to_find) | col("policy_number").isin(identifiers_to_find))
            .select("policy_number")
            .distinct()
        )

        found = [r.policy_number for r in result.collect()]

        return found

    def get_debit_order_info(self, policy_number: str) -> Dict[str, Any]:

        spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
        df = spark.table("classic_demo.insurance_customer_service.policies")
        policy = df.filter(col("policy_number") == policy_number).collect()[0]

        df = spark.table("classic_demo.insurance_customer_service.debit_orders")
        debit_info = df.filter(col("policy_number") == policy_number).collect()[0]

        if not debit_info:
            return {"success": False, "message": f"No debit order instruction found for policy '{policy_number}'."}

        return {
            "success": True,
            "policy_number": policy_number,
            "account_holder": debit_info["account_holder"],
            "bank_name": debit_info["bank_name"],
            "account_number_masked": debit_info["account_number_masked"],
            "branch_code": debit_info["branch_code"],
            "debit_day": f"Day {debit_info['debit_day']} of each month",
            "debit_amount": f"{policy['currency']} {debit_info['debit_amount']:.2f}",
            "last_successful_debit_date": debit_info["last_successful_debit_date"],
            "last_debit_status": debit_info["last_debit_status"],
            "next_debit_date": debit_info["next_debit_date"],
            "payment_frequency": debit_info["payment_frequency"],
            "message": f"Debit order for policy {policy_number} is scheduled for day {debit_info['debit_day']} of every month at R {debit_info['debit_amount']:.2f}."
        }

    def get_policy_summary(self, policy_number: str) -> Dict[str, Any]:

        spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
        df = spark.table("classic_demo.insurance_customer_service.policies")
        policy = df.filter(col("policy_number") == policy_number).collect()[0]
        if not policy:
            return {"success": False, "message": f"Policy '{policy_number}' not found."}
        
        # df = spark.table("classic_demo.insurance_customer_service.policies")
        # cust = df.filter(col("customer_id") == policy.customer_id).collect()[0]

        df = spark.table("classic_demo.insurance_customer_service.vehicle_noted_interest")
        vehicles = df.filter(col("policy_number") == policy_number).collect()

        return {
            "success": True,
            "policy_number": policy_number,
            # "policyholder_name": f"{cust.get('first_name', '')} {cust.get('last_name', '')}",
            "product_type": policy["product_type"],
            "status": policy["status"],
            "start_date": policy["start_date"],
            "premium_amount": f"{policy['currency']} {policy['premium_amount']:.2f}",
            "basic_excess": f"{policy['currency']} {policy['excess_amount']:.2f}",
            "cover_details": policy["cover_details"],
            "noted_vehicles": [f"{v['year']} {v['make']} {v['model']} ({v['registration_number']})" for v in vehicles] if vehicles else ["None"]
        }

    def get_summary(self, possible_policy_numbers: List[str]) -> str:

        if possible_policy_numbers:
            summary = ""
            for policy in possible_policy_numbers:
                summary += f"\n#### Policy: {policy}"
                summary += f"\n{self.get_policy_summary(policy)}"
                summary += f"\n\n{self.get_debit_order_info(policy)}"
                summary += "\n\n================================================="
        
        else:
            summary = 'No policies could be identified at the moment - waiting for the system to respond'

        return summary

policy_extractor = PolicyExtractor()
